# Pistachio Cultivar Classification — v5.1 add-on: robustness to acquisition conditions and training cost

This notebook extends the v5.1 analysis **without re-running it**. It reuses the cached features, the
group-aware splits and the fine-tuned checkpoints saved by `pistachio_classification_torch_v5_1.ipynb`
in `RESULTS_DIR`, and writes its outputs to `RESULTS_DIR/addon/`.

1. **Robustness test.** The test images of every seed are modified in a controlled way (illumination,
   contrast, color cast, blur, noise, JPEG compression, rotation and background replacement). The
   selected hybrid model (trained on unmodified training images) and the end-to-end fine-tuned CNNs are
   evaluated on each modified test set. This approximates acquisition conditions that differ from the
   original dataset. Nothing is retrained.
2. **Training cost.** Wall-clock training time of the hybrid pipeline (feature extraction + scaler + PCA +
   SVM) is compared with end-to-end fine-tuning (time of one training epoch × number of epochs).

Runtime on an RTX 3050 laptop GPU: about 15–20 minutes.

In [ ]:
import os, json, time, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
import io

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
from scipy import ndimage
from skimage.filters import threshold_otsu

warnings.filterwarnings('ignore')

RESULTS_DIR = 'results_v5_1'          # output folder of the v5.1 notebook
with open(os.path.join(RESULTS_DIR, 'environment.json')) as f:
    CONFIG = json.load(f)['config']
ADDON = os.path.join(RESULTS_DIR, 'addon')
os.makedirs(ADDON, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS = CONFIG['seeds']
MAIN_SEED = CONFIG['main_seed']
IMG_SIZE = CONFIG['img_size']
print(f'Results folder: {os.path.abspath(RESULTS_DIR)} | device: {DEVICE} | seeds: {SEEDS}')


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 1. Data, splits, features and selected model (identical to v5.1)

In [ ]:
file_df = pd.read_csv(os.path.join(RESULTS_DIR, 'splits', 'file_list.csv'))
split_df = pd.read_csv(os.path.join(RESULTS_DIR, 'splits', 'split_indices.csv'))
DATA_DIR = CONFIG['data_dir']
PATHS = np.array([os.path.join(DATA_DIR, f) for f in file_df['file']])
LABELS = file_df['label'].values
N = len(PATHS)
assert (split_df['index'].values == np.arange(N)).all()
SPLITS = {s: {p: np.where(split_df[f'seed_{s}'].values == p)[0] for p in ['train', 'val', 'test']} for s in SEEDS}

cv_sum = pd.read_csv(os.path.join(RESULTS_DIR, 'tables', 'table_cv_all18_summary.csv'))
BEST_KEY = cv_sum.sort_values('cv_acc_mean', ascending=False).iloc[0]['model']
BEST_MOD, BEST_CNN, BEST_CLF = BEST_KEY.split('+')
print(f'Selected hybrid model (from v5.1 CV): {BEST_KEY}')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
_MEAN_T = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
_STD_T = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)


def load_resized(path, size):
    with Image.open(path) as im:
        return np.asarray(im.convert('RGB').resize((size, size), Image.BILINEAR))


IMG_RGB = np.stack([load_resized(p, IMG_SIZE) for p in tqdm(PATHS, desc='Loading images')])
IMG_GRAY = np.stack([np.asarray(Image.fromarray(a).convert('L')) for a in IMG_RGB])


def pistachio_mask(gray):
    thr = threshold_otsu(gray)
    border = np.concatenate([gray[0], gray[-1], gray[:, 0], gray[:, -1]])
    fg = gray > thr if border.mean() < thr else gray < thr
    fg = ndimage.binary_closing(fg, iterations=2)
    lab, n = ndimage.label(fg)
    if n > 1:
        sizes_ = ndimage.sum(fg, lab, range(1, n + 1))
        fg = lab == (np.argmax(sizes_) + 1)
    return ndimage.binary_fill_holes(fg)


MASKS = np.stack([pistachio_mask(g) for g in IMG_GRAY])


def arrays_to_tensor(arr):
    t = torch.from_numpy(np.ascontiguousarray(arr)).permute(0, 3, 1, 2).float().div_(255.0)
    return (t - _MEAN_T) / _STD_T


def to_modality(arr, modality):
    if modality == 'RGB':
        return arr
    g = np.stack([np.asarray(Image.fromarray(a).convert('L')) for a in arr])
    return np.repeat(g[..., None], 3, axis=-1)


def build_backbone(name, pretrained=True):
    if name == 'ResNet50':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        dim = m.fc.in_features; m.fc = nn.Identity()
    elif name == 'EfficientNetB0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
        dim = m.classifier[-1].in_features; m.classifier = nn.Identity()
    else:
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
        dim = m.classifier[-1].in_features; m.classifier = nn.Identity()
    return m, dim


class FineTuneNet(nn.Module):
    def __init__(self, cnn, pretrained=True):
        super().__init__()
        self.backbone, dim = build_backbone(cnn, pretrained)
        self.head = nn.Linear(dim, 2)

    def forward(self, x):
        return self.head(self.backbone(x))


def make_pipeline(seed):
    return Pipeline([('scaler', StandardScaler()),
                     ('pca', PCA(n_components=CONFIG['pca_var'], svd_solver='full', random_state=seed)),
                     ('clf', SVC(kernel='rbf', C=CONFIG['svm_C'], gamma=CONFIG['svm_gamma'], random_state=seed))])


@torch.no_grad()
def extract(model, arr, bs=None):
    bs = bs or CONFIG['batch_size']
    out = []
    for s in range(0, len(arr), bs):
        out.append(model(arrays_to_tensor(arr[s:s + bs]).to(DEVICE)).float().cpu().numpy())
    return np.concatenate(out)


backbone, _ = build_backbone(BEST_CNN, CONFIG['pretrained'])
backbone = backbone.eval().to(DEVICE)
FEATS = np.load(os.path.join(RESULTS_DIR, 'features', f'{BEST_CNN}_{BEST_MOD}.npy'))
check_idx = np.arange(min(64, N))
diff = np.abs(extract(backbone, to_modality(IMG_RGB[check_idx], BEST_MOD)) - FEATS[check_idx]).max()
print(f'Max difference between recomputed and cached features: {diff:.2e} (should be ~0)')
assert diff < 1e-2, 'Cached features do not match: check RESULTS_DIR and data_dir.'

PIPES = {s: make_pipeline(s).fit(FEATS[SPLITS[s]['train']], LABELS[SPLITS[s]['train']]) for s in SEEDS}
clean_acc = {s: accuracy_score(LABELS[SPLITS[s]['test']], PIPES[s].predict(FEATS[SPLITS[s]['test']])) for s in SEEDS}
print('Clean test accuracy per seed (should equal v5.1):', {s: round(a * 100, 2) for s, a in clean_acc.items()})

## 2. Controlled perturbations of the test images
Each perturbation is applied to the 224 × 224 RGB test image before the network input. Background
replacement uses the pistachio mask (Otsu threshold) and replaces the black background with a uniform
gray or white background.

In [ ]:
def jpeg(img, q):
    buf = io.BytesIO(); img.save(buf, format='JPEG', quality=q); buf.seek(0)
    return Image.open(buf).convert('RGB')


def color_cast(arr, r, b):
    a = arr.astype(float); a[..., 0] *= r; a[..., 2] *= b
    return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))


def gaussian_noise(arr, sigma, seed):
    rng = np.random.default_rng(seed)
    return Image.fromarray(np.clip(arr.astype(float) + rng.normal(0, sigma, arr.shape), 0, 255).astype(np.uint8))


def background(arr, mask, value):
    a = arr.copy(); a[~mask] = value
    return Image.fromarray(a)


PERTURBATIONS = {
    'Original':                 ('Reference', lambda a, k: Image.fromarray(a)),
    'Brightness −40 %':         ('Illumination', lambda a, k: ImageEnhance.Brightness(Image.fromarray(a)).enhance(0.6)),
    'Brightness −20 %':         ('Illumination', lambda a, k: ImageEnhance.Brightness(Image.fromarray(a)).enhance(0.8)),
    'Brightness +20 %':         ('Illumination', lambda a, k: ImageEnhance.Brightness(Image.fromarray(a)).enhance(1.2)),
    'Brightness +40 %':         ('Illumination', lambda a, k: ImageEnhance.Brightness(Image.fromarray(a)).enhance(1.4)),
    'Contrast −30 %':           ('Illumination', lambda a, k: ImageEnhance.Contrast(Image.fromarray(a)).enhance(0.7)),
    'Contrast +30 %':           ('Illumination', lambda a, k: ImageEnhance.Contrast(Image.fromarray(a)).enhance(1.3)),
    'Warm color cast (R+10 %, B−10 %)': ('Color', lambda a, k: color_cast(a, 1.10, 0.90)),
    'Cool color cast (R−10 %, B+10 %)': ('Color', lambda a, k: color_cast(a, 0.90, 1.10)),
    'Gaussian blur (σ = 1 px)': ('Camera', lambda a, k: Image.fromarray(a).filter(ImageFilter.GaussianBlur(1))),
    'Gaussian blur (σ = 2 px)': ('Camera', lambda a, k: Image.fromarray(a).filter(ImageFilter.GaussianBlur(2))),
    'Gaussian noise (σ = 10)':  ('Camera', lambda a, k: gaussian_noise(a, 10, k)),
    'JPEG quality 30':          ('Camera', lambda a, k: jpeg(Image.fromarray(a), 30)),
    'Rotation 90°':             ('Orientation', lambda a, k: Image.fromarray(a).rotate(90)),
    'Rotation 180°':            ('Orientation', lambda a, k: Image.fromarray(a).rotate(180)),
    'Gray background (128)':    ('Background', lambda a, k: background(a, MASKS[k], 128)),
    'White background (255)':   ('Background', lambda a, k: background(a, MASKS[k], 255)),
}

TEST_ALL = np.unique(np.concatenate([SPLITS[s]['test'] for s in SEEDS]))
PERT_IMG = {}
for name, (_, fn) in tqdm(PERTURBATIONS.items(), desc='Building perturbed test images'):
    PERT_IMG[name] = {k: np.asarray(fn(IMG_RGB[k], k).convert('RGB')) for k in TEST_ALL}

ex = TEST_ALL[0]
fig, axes = plt.subplots(2, 9, figsize=(20, 5))
for ax, name in zip(axes.ravel(), PERTURBATIONS):
    ax.imshow(PERT_IMG[name][ex]); ax.set_title(name, fontsize=7); ax.axis('off')
for ax in axes.ravel()[len(PERTURBATIONS):]:
    ax.axis('off')
plt.suptitle('Controlled perturbations applied to the test images (one example)', fontweight='bold')
plt.tight_layout(); plt.savefig(os.path.join(ADDON, 'fig_perturbation_examples.png'), dpi=300, bbox_inches='tight'); plt.show()

## 3. Robustness of the selected hybrid model and of the fine-tuned CNNs

In [ ]:
rows = []
# Hybrid model: features of every perturbed test image are extracted once, then each seed's pipeline is applied.
for name in tqdm(PERTURBATIONS, desc='Hybrid model'):
    arr = to_modality(np.stack([PERT_IMG[name][k] for k in TEST_ALL]), BEST_MOD)
    F = extract(backbone, arr)
    pos = {k: i for i, k in enumerate(TEST_ALL)}
    for s in SEEDS:
        te = SPLITS[s]['test']; Xt = F[[pos[k] for k in te]]
        yhat = PIPES[s].predict(Xt)
        rows.append({'model': f'Hybrid {BEST_KEY}', 'perturbation': name, 'group': PERTURBATIONS[name][0], 'seed': s,
                     'acc': accuracy_score(LABELS[te], yhat), 'f1_macro': f1_score(LABELS[te], yhat, average='macro'),
                     'mcc': matthews_corrcoef(LABELS[te], yhat)})

# Fine-tuned CNNs (RGB), checkpoints from v5.1
FT_MODELS = CONFIG.get('ft_models', [])
ft_seeds = SEEDS if CONFIG.get('ft_seeds') is None else CONFIG['ft_seeds']
for cnn in FT_MODELS:
    for s in ft_seeds:
        ck = os.path.join(RESULTS_DIR, 'checkpoints', f'finetuned_{cnn}_seed{s}.pt')
        if not os.path.exists(ck):
            print('missing checkpoint:', ck); continue
        net = FineTuneNet(cnn, pretrained=False)
        sd = torch.load(ck, map_location='cpu')
        # v5.1 checkpoints also store the Grad-CAM layer reference ('target_layer.*'); it duplicates backbone weights.
        sd = {k: v for k, v in sd.items() if not k.startswith('target_layer.')}
        net.load_state_dict(sd, strict=True)
        net = net.eval().to(DEVICE)
        te = SPLITS[s]['test']
        for name in PERTURBATIONS:
            with torch.no_grad():
                logits = extract(net, np.stack([PERT_IMG[name][k] for k in te]))
            yhat = logits.argmax(1)
            rows.append({'model': f'Fine-tuned {cnn}', 'perturbation': name, 'group': PERTURBATIONS[name][0], 'seed': s,
                         'acc': accuracy_score(LABELS[te], yhat), 'f1_macro': f1_score(LABELS[te], yhat, average='macro'),
                         'mcc': matthews_corrcoef(LABELS[te], yhat)})
        del net
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

ROB = pd.DataFrame(rows)
ROB.to_csv(os.path.join(ADDON, 'raw_robustness.csv'), index=False)
ref = ROB[ROB.perturbation == 'Original'].set_index(['model', 'seed']).acc
ROB['drop_pp'] = [(ref.loc[(m, s)] - a) * 100 for m, s, a in zip(ROB.model, ROB.seed, ROB.acc)]
summ = ROB.groupby(['model', 'group', 'perturbation'], sort=False).agg(
    acc_mean=('acc', 'mean'), acc_sd=('acc', 'std'), drop_pp_mean=('drop_pp', 'mean'), drop_pp_sd=('drop_pp', 'std'),
    mcc_mean=('mcc', 'mean'), n_seeds=('acc', 'size')).reset_index()
summ['acc_mean'] *= 100; summ['acc_sd'] *= 100
summ.round(3).to_csv(os.path.join(ADDON, 'table_robustness_summary.csv'), index=False, encoding='utf-8-sig')
wide = summ.pivot_table(index=['group', 'perturbation'], columns='model', values='acc_mean', sort=False).round(2)
wide.to_csv(os.path.join(ADDON, 'table_robustness_accuracy_wide.csv'), encoding='utf-8-sig')
print('Test accuracy (%) under each perturbation, mean over seeds:')
print(wide.to_string())

fig, ax = plt.subplots(figsize=(13, 6))
order = list(PERTURBATIONS)
for m in summ.model.unique():
    s_ = summ[summ.model == m].set_index('perturbation').loc[order]
    ax.errorbar(range(len(order)), s_.acc_mean, yerr=s_.acc_sd.fillna(0), fmt='o-', capsize=3, label=m)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Test accuracy (%), mean ± SD over seeds'); ax.legend(fontsize=8); ax.grid(alpha=0.3)
ax.set_title('Robustness to controlled changes in acquisition conditions (no retraining)')
plt.tight_layout(); plt.savefig(os.path.join(ADDON, 'fig_robustness.png'), dpi=300, bbox_inches='tight'); plt.show()

## 4. Training cost: hybrid pipeline vs. end-to-end fine-tuning
Hybrid: one forward pass over the training images (feature extraction) + fitting scaler, PCA and SVM.
Fine-tuning: wall-clock time of one training epoch with the v5.1 settings (augmentation, AdamW, mixed
precision on GPU), multiplied by the configured number of epochs. Seed 42, same training set.

In [ ]:
def sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()


AUG_TF = T.Compose([T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(15),
                    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1)])
TO_TENSOR = T.Compose([T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


class TrainDS(Dataset):
    def __init__(self, idx):
        self.idx = idx

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        j = self.idx[i]
        return TO_TENSOR(AUG_TF(Image.fromarray(IMG_RGB[j]))), int(LABELS[j])


tr = SPLITS[MAIN_SEED]['train']
cost = []
for cnn in sorted(set(FT_MODELS) | {BEST_CNN}):
    m, _ = build_backbone(cnn, CONFIG['pretrained']); m = m.eval().to(DEVICE)
    sync(); t0 = time.perf_counter()
    Ftr = extract(m, to_modality(IMG_RGB[tr], 'RGB')); sync(); t_feat = time.perf_counter() - t0
    t0 = time.perf_counter(); make_pipeline(MAIN_SEED).fit(Ftr, LABELS[tr]); t_fit = time.perf_counter() - t0
    del m
    row = {'CNN': cnn, 'n_train': len(tr), 'hybrid_feature_extraction_s': t_feat, 'hybrid_scaler_pca_svm_fit_s': t_fit,
           'hybrid_total_s': t_feat + t_fit}
    if cnn in FT_MODELS:
        set_seed(MAIN_SEED)
        net = FineTuneNet(cnn, CONFIG['pretrained']).to(DEVICE)
        use_amp = DEVICE.type == 'cuda'
        try:
            scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
        except (AttributeError, TypeError):
            scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
        opt = torch.optim.AdamW([{'params': net.backbone.parameters(), 'lr': CONFIG['ft_lr_backbone']},
                                 {'params': net.head.parameters(), 'lr': CONFIG['ft_lr_head']}],
                                weight_decay=CONFIG['ft_weight_decay'])
        g = torch.Generator(); g.manual_seed(MAIN_SEED)
        loader = DataLoader(TrainDS(tr), batch_size=CONFIG['ft_batch_size'], shuffle=True, num_workers=0, generator=g)
        crit = nn.CrossEntropyLoss(); net.train()
        sync(); t0 = time.perf_counter()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
                loss = crit(net(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sync(); t_ep = time.perf_counter() - t0
        trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
        row.update({'finetune_trainable_params_M': trainable / 1e6, 'finetune_epoch_s': t_ep,
                    'finetune_epochs': CONFIG['ft_epochs'], 'finetune_total_s_est': t_ep * CONFIG['ft_epochs'],
                    'finetune_over_hybrid_ratio': t_ep * CONFIG['ft_epochs'] / (t_feat + t_fit)})
        del net, opt
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    cost.append(row)
COST = pd.DataFrame(cost)
COST.round(3).to_csv(os.path.join(ADDON, 'table_training_cost.csv'), index=False)
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'} | training images: {len(tr)}")
print(COST.round(2).to_string(index=False))
print(f'\nAll add-on outputs saved under: {os.path.abspath(ADDON)}')